In [1]:
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes
import os

def get_cipher(mode):
    # AES-128 requires a 16-byte key
    key = b'sixteen_byte_key' 
    return Cipher(algorithms.AES(key), mode)

def encrypt(plaintext, mode):
    encryptor = get_cipher(mode).encryptor()
    return encryptor.update(plaintext) + encryptor.finalize()

def decrypt(ciphertext, mode):
    decryptor = get_cipher(mode).decryptor()
    return decryptor.update(ciphertext) + decryptor.finalize()

def flip_bit(byte_data, byte_index, bit_index=0):
    """Flips a single bit in a bytearray."""
    data_array = bytearray(byte_data)
    data_array[byte_index] ^= (1 << bit_index)
    return bytes(data_array)

# --- SETUP ---
# Two identical blocks of 16 bytes
block1 = b'SAME_PLAINTEXT16'
block2 = b'SAME_PLAINTEXT16'
plaintext = block1 + block2 

print("--- 1. IDENTICAL PLAINTEXT TEST ---")
# ECB Mode
ct_ecb = encrypt(plaintext, modes.ECB())
print(f"[ECB] Block 1 CT: {ct_ecb[:16].hex()}")
print(f"[ECB] Block 2 CT: {ct_ecb[16:].hex()}")
print(f"Are ECB ciphertexts identical? {ct_ecb[:16] == ct_ecb[16:]}\n")

# CBC Mode
iv = os.urandom(16)
ct_cbc = encrypt(plaintext, modes.CBC(iv))
print(f"[CBC] Block 1 CT: {ct_cbc[:16].hex()}")
print(f"[CBC] Block 2 CT: {ct_cbc[16:].hex()}")
print(f"Are CBC ciphertexts identical? {ct_cbc[:16] == ct_cbc[16:]}\n")


print("--- 2. ERROR PROPAGATION TEST (1-bit error in Block 1) ---")
# Let's corrupt the very first byte of the first ciphertext block
corrupted_ct_ecb = flip_bit(ct_ecb, byte_index=0)
corrupted_ct_cbc = flip_bit(ct_cbc, byte_index=0)

# Decrypting ECB
dec_ecb = decrypt(corrupted_ct_ecb, modes.ECB())
print(f"[ECB Decryption corrupted CT]")
print(f"Decrypted Block 1: {dec_ecb[:16]}")
print(f"Decrypted Block 2: {dec_ecb[16:]} (Unchanged!)\n")

# Decrypting CBC
dec_cbc = decrypt(corrupted_ct_cbc, modes.CBC(iv))
print(f"[CBC Decryption corrupted CT]")
print(f"Decrypted Block 1: {dec_cbc[:16]} (Garbled!)")
print(f"Decrypted Block 2: {dec_cbc[16:]} (Exactly 1 bit flipped!)")

--- 1. IDENTICAL PLAINTEXT TEST ---
[ECB] Block 1 CT: b47f63ca48880566fe320b48715dc4a5
[ECB] Block 2 CT: b47f63ca48880566fe320b48715dc4a5
Are ECB ciphertexts identical? True

[CBC] Block 1 CT: 423248dfa40d26daee4e76039238e4cf
[CBC] Block 2 CT: 8860b07ec56f38e146a07938460ab607
Are CBC ciphertexts identical? False

--- 2. ERROR PROPAGATION TEST (1-bit error in Block 1) ---
[ECB Decryption corrupted CT]
Decrypted Block 1: b'\x96\x02\x13R<\x8e\xa6"O n\xa9\x14\x05\xc3\xcb'
Decrypted Block 2: b'SAME_PLAINTEXT16' (Unchanged!)

[CBC Decryption corrupted CT]
Decrypted Block 1: b'\x071\x0b\xd8\xd7\xe0\x1c\xe9\xf6:s\xcf\xb8\x96\xa33' (Garbled!)
Decrypted Block 2: b'RAME_PLAINTEXT16' (Exactly 1 bit flipped!)
